### Preparación del Entorno
Importa las librerías necesarias y carga el dataset

In [2]:
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base
from sqlalchemy.orm import sessionmaker

# Cargar dataset de ventas online
df = pd.read_csv('../data/hotel_bookings.csv', encoding='ISO-8859-1')


df.head(10) 

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
6,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,No Deposit,NaN,NaN,0,Transient,107.0,0,0,Check-Out,2015-07-03
7,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,No Deposit,303.0,NaN,0,Transient,103.0,0,1,Check-Out,2015-07-03
8,Resort Hotel,1,85,2015,July,27,1,0,3,2,...,No Deposit,240.0,NaN,0,Transient,82.0,0,1,Canceled,2015-05-06
9,Resort Hotel,1,75,2015,July,27,1,0,3,2,...,No Deposit,15.0,NaN,0,Transient,105.5,0,0,Canceled,2015-04-22


### Imputacion de datos
Buscar datos duplicados y nulos

In [3]:
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [4]:
# Imputar valores faltantes
df["children"] = df["children"].fillna(df["children"].median()) # mediana
df["country"] = df["country"].fillna(df["country"].mode()[0])   # moda
df["agent"] = df["agent"].fillna(0)                             # cero
df["company"] = df["company"].fillna(0)                         # cero

df.to_csv('../data/hotel_bookings_imputed.csv', index=False)

df.isna().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
company                           0
days_in_waiting_list              0
customer_type                     0
adr                         

### Definición del Esquema ORM

In [5]:
from sqlalchemy import Date

Base = declarative_base()

class Hotel(Base):
    __tablename__ = 'hotels'
    hotel_id = Column(Integer, primary_key=True, autoincrement=True)
    hotel_type = Column(String, unique=True, nullable=False)

class CustomerInfo(Base):
    __tablename__ = 'customers'
    customer_id = Column(Integer, primary_key=True, autoincrement=True)
    customer_type = Column(String, nullable=False)
    country = Column(String)


class Reservation(Base):
    __tablename__ = 'reservations'

    res_id = Column(Integer, primary_key=True, autoincrement=True)

    hotel_id = Column(Integer, ForeignKey('hotels.hotel_id'), nullable=False)
    customer_id = Column(Integer, ForeignKey('customers.customer_id'), nullable=False)

    lead_time = Column(Integer)
    adr = Column(Float)
    is_canceled = Column(Integer)

    arrival_date = Column(Date)

    stays_in_weekend_nights = Column(Integer)
    stays_in_week_nights = Column(Integer)

    adults = Column(Integer)
    children = Column(Integer)
    babies = Column(Integer)

    previous_cancellations = Column(Integer)
    total_of_special_requests = Column(Integer)
    deposit_type = Column(String)

print(Hotel.__table__)
print(CustomerInfo.__table__)
print(Reservation.__table__)

hotels
customers
reservations


### Creación de la BD y Población

In [8]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from datetime import datetime
import pandas as pd



# ENGINE + SESSION
engine = create_engine('sqlite:///../db/hotel_local.db', echo=False, future=True)
Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()

try:
    # LIMPIEZA
    df_clean = pd.read_csv('../data/hotel_bookings_imputed.csv', encoding='ISO-8859-1')

    df_clean['arrival_date'] = pd.to_datetime(
        df_clean['arrival_date_year'].astype(str) + '-' +
        df_clean['arrival_date_month'] + '-' +
        df_clean['arrival_date_day_of_month'].astype(str),
        format="%Y-%B-%d",
        errors='coerce'
    )

    df_clean = df_clean.dropna(subset=['arrival_date'])

    # CREAR CLIENTE DUMMY
    dummy_customer = session.query(CustomerInfo).first()

    if not dummy_customer:
        dummy_customer = CustomerInfo(customer_type="Generic Customer", country=None)
        session.add(dummy_customer)
        session.commit()

    customer_id = dummy_customer.customer_id

    # INSERTAR HOTELES
    existing_hotels = {
        h.hotel_type for h in session.query(Hotel).all()
    }

    new_hotels = [
        Hotel(hotel_type=h)
        for h in df_clean['hotel'].unique()
        if h not in existing_hotels
    ]

    session.add_all(new_hotels)
    session.commit()

    print(f"Hoteles insertados: {len(new_hotels)}")


    # MAPEAR HOTEL → ID
    hotel_map = {
        h.hotel_type: h.hotel_id
        for h in session.query(Hotel).all()
    }

    # INSERTAR RESERVAS
    batch_size = 5000
    total_inserted = 0

    for i in range(0, len(df_clean), batch_size):
        batch = df_clean.iloc[i:i+batch_size]
        reservations = []

        for _, row in batch.iterrows():
            hotel_id = hotel_map.get(row['hotel'])
            if hotel_id is None:
                continue

            reservations.append(
                Reservation(
                    hotel_id=hotel_id,
                    customer_id=customer_id,  
                    lead_time=int(row['lead_time']),
                    adr=float(row['adr']),
                    is_canceled=int(row['is_canceled']),
                    arrival_date=row['arrival_date'].date(),
                    stays_in_weekend_nights=int(row["stays_in_weekend_nights"]),
                    stays_in_week_nights=int(row["stays_in_week_nights"]),
                    adults=int(row["adults"]),
                    children=int(row["children"]),
                    babies=int(row["babies"]),
                    previous_cancellations=int(row['previous_cancellations']),
                    total_of_special_requests=int(row['total_of_special_requests']),
                    deposit_type=str(row['deposit_type']),
                )
            )

        session.add_all(reservations)
        session.commit()

        total_inserted += len(reservations)
        print(f"Batch {i} - {i+len(batch)} insertado")

    print(f"Total reservas insertadas: {total_inserted}")

except Exception as e:
    session.rollback()
    print("Error:", e)

finally:
    session.close()

Hoteles insertados: 2
Batch 0 - 5000 insertado
Batch 5000 - 10000 insertado
Batch 10000 - 15000 insertado
Batch 15000 - 20000 insertado
Batch 20000 - 25000 insertado
Batch 25000 - 30000 insertado
Batch 30000 - 35000 insertado
Batch 35000 - 40000 insertado
Batch 40000 - 45000 insertado
Batch 45000 - 50000 insertado
Batch 50000 - 55000 insertado
Batch 55000 - 60000 insertado
Batch 60000 - 65000 insertado
Batch 65000 - 70000 insertado
Batch 70000 - 75000 insertado
Batch 75000 - 80000 insertado
Batch 80000 - 85000 insertado
Batch 85000 - 90000 insertado
Batch 90000 - 95000 insertado
Batch 95000 - 100000 insertado
Batch 100000 - 105000 insertado
Batch 105000 - 110000 insertado
Batch 110000 - 115000 insertado
Batch 115000 - 119390 insertado
Total reservas insertadas: 119390


### CRUD querys usando sqlite3 nativo

In [9]:
import sqlite3
import pandas as pd

# Conectamos usando sqlite3 nativo para operaciones crudas seguras
conn = sqlite3.connect('../db/hotel_local.db')
cursor = conn.cursor()

# --- OPERACIONES CRUD 

# 1. INSERT: Creamos un registro "Dummy" 
cursor.execute("""
    INSERT INTO reservations (hotel_id, customer_id, lead_time, adr, is_canceled)
    VALUES (?, ?, ?, ?, ?)
""", (1, 1, 9999, 0.0, 1))

# Capturamos el ID de ese registro falso que acabamos de crear
dummy_id = cursor.lastrowid

# 2. UPDATE: Actualizamos solo el registro falso usando WHERE y ?
cursor.execute("UPDATE reservations SET adr = ? WHERE res_id = ?", (999.99, dummy_id))

# 3. DELETE: Borramos únicamente el registro falso para no afectar el dataset real de ML
cursor.execute("DELETE FROM reservations WHERE res_id = ?", (dummy_id,))

conn.commit()
print("Operaciones CRUD (Insert, Update, Delete) ejecutadas.")

# Consulta SQL avanzada con JOIN, subconsulta, GROUP BY, HAVING y ORDER BY
query_avanzada = """
SELECT 
    h.hotel_type,
    COUNT(r.res_id) as total_reservas,
    AVG(r.adr) as adr_promedio
FROM hotels h
INNER JOIN reservations r ON h.hotel_id = r.hotel_id
LEFT JOIN customers c ON r.customer_id = c.customer_id
WHERE r.lead_time > (SELECT AVG(lead_time) FROM reservations)
GROUP BY h.hotel_type
HAVING total_reservas > 100 
ORDER BY adr_promedio DESC;
"""

df_sql_avanzado = pd.read_sql_query(query_avanzada, conn)
conn.close()

print("\n--- Resultados de la Consulta SQL Avanzada ---")
display(df_sql_avanzado)

Operaciones CRUD (Insert, Update, Delete) ejecutadas.

--- Resultados de la Consulta SQL Avanzada ---


,hotel_type,total_reservas,adr_promedio
0,City Hotel,31236,101.592121
1,Resort Hotel,14526,98.901794
